# Mamba-JEPA v1 hard-disjoint VICReg — all positions — 12x12 Colab Training

This run uses `configs/mamba_jepa_v1_hd_vicreg_b12_all_positions.yml`. The launcher loads it automatically.

Run the cells in order. Run the full-training cell only after the smoke gate passes.

## 1. Mount Drive and install the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys

PROJECT_ROOT = Path('/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/Master_Thesis_Artifacts')
DATA_DIR = ARTIFACT_ROOT / 'data' / 'othello_12x12' / 'corpus_v1'

assert (PROJECT_ROOT / 'scripts' / 'launch_12x12_training.py').is_file(), PROJECT_ROOT
assert (DATA_DIR / 'manifest.json').is_file(), DATA_DIR
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR    :', DATA_DIR)

def run_streaming(command):
    # Forward child output as it arrives, including '\r'-refreshed progress
    # updates, so the trainer's in-place chunk progress bar renders live.
    # Line-based iteration would buffer the bar until the next newline.
    import codecs
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print('COMMAND:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(
        list(map(str, command)),
        cwd=PROJECT_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    while True:
        data = process.stdout.read1(8192)
        if not data:
            break
        sys.stdout.write(decoder.decode(data))
        sys.stdout.flush()
    tail = decoder.decode(b'', True)
    if tail:
        sys.stdout.write(tail)
        sys.stdout.flush()
    returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode


Mounted at /content/drive
PROJECT_ROOT: /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code
DATA_DIR    : /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_12x12/corpus_v1


## 2. Install and verify the official Mamba CUDA backend

In [ ]:
import importlib.util
if importlib.util.find_spec('mamba_ssm') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'packaging', 'ninja'], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        'mamba-ssm[causal-conv1d]', '--no-build-isolation'
    ], check=True)

import torch
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'
import mamba_ssm
print('GPU:', torch.cuda.get_device_name(0))
print('mamba_ssm ready')


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
mamba_ssm ready


## 3. Validate the exact corpus and split

In [ ]:
import json
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
expected = {
    'complete': True, 'board_size': 12, 'generated_games': 23_800_000,
    'training_games': 20_000_000, 'heldout_games': 3_800_000,
    'model_block_size': 139,
}
for key, value in expected.items():
    assert manifest.get(key) == value, (key, manifest.get(key), value)
assert len(manifest['shards']) == 238
print('12x12 corpus gate: PASS')


12x12 corpus gate: PASS


## 4. Smoke gate — one training shard

This writes to a separate `_smoke` run and cannot overwrite the full run.

In [ ]:
print('STARTING smoke: mamba_jepa_allpos', flush=True)
run_streaming([sys.executable, '-u', 'scripts/launch_12x12_training.py', 'mamba_jepa_allpos', '--smoke'])
print('SMOKE FINISHED: mamba_jepa_allpos', flush=True)


STARTING smoke: mamba_jepa_allpos
COMMAND: /usr/bin/python3 -u scripts/launch_12x12_training.py mamba_jepa_allpos --smoke
[12x12 launcher] START experiment=mamba_jepa_allpos smoke=True
[12x12 launcher] validating corpus manifest...
[12x12 launcher] corpus validation: PASS
[12x12 launcher] command: /usr/bin/python3 -u -m othello_research.training.train_mamba_jepa --config /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code/configs/mamba_jepa_v1_hd_vicreg_b12_all_positions.yml --data_dir /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_12x12/corpus_v1 --li_split_train_chunks 200 --out_dir /content/runs/Mamba-JEPA/mamba_jepa_v1_hd_vicreg_allpos_b12_smoke --drive_sync_dir /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-JEPA/mamba_jepa_v1_hd_vicreg_allpos_b12_smoke --drive_sync_every_chunks 5 --board_size 12 --batch_size 256 --precision bf16 --passes_over_data 1 --checkpoint_every_chunks 0 --milestone_games 1000000,5000000,10000000,15000000,20000000 --seed 42 --ma

## 5. Full 20M-game training

Run this cell only after smoke loss is finite and there was no OOM.

In [5]:
print('STARTING full training: mamba_jepa_allpos', flush=True)
run_streaming([sys.executable, '-u', 'scripts/launch_12x12_training.py', 'mamba_jepa_allpos'])
print('FULL TRAINING FINISHED: mamba_jepa_allpos', flush=True)


STARTING full training: mamba_jepa_allpos
COMMAND: /usr/bin/python3 -u scripts/launch_12x12_training.py mamba_jepa_allpos
[12x12 launcher] START experiment=mamba_jepa_allpos smoke=False
[12x12 launcher] validating corpus manifest...
[12x12 launcher] corpus validation: PASS
[12x12 launcher] command: /usr/bin/python3 -u -m othello_research.training.train_mamba_jepa --config /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code/configs/mamba_jepa_v1_hd_vicreg_b12_all_positions.yml --data_dir /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_12x12/corpus_v1 --li_split_train_chunks 200 --out_dir /content/runs/Mamba-JEPA/mamba_jepa_v1_hd_vicreg_allpos_b12 --drive_sync_dir /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-JEPA/mamba_jepa_v1_hd_vicreg_allpos_b12 --drive_sync_every_chunks 5 --board_size 12 --batch_size 256 --precision bf16 --passes_over_data 1 --checkpoint_every_chunks 0 --milestone_games 1000000,5000000,10000000,15000000,20000000 --seed 42
[12x12 launcher

## 6. Resume after a disconnected runtime (optional)

Set `RESUME_CHECKPOINT` to a real synced checkpoint, then run the cell.

In [6]:
RESUME_CHECKPOINT = None  # e.g. '/content/drive/MyDrive/.../latest.pt'
if RESUME_CHECKPOINT:
    run_streaming([sys.executable, '-u', 'scripts/launch_12x12_training.py', 'mamba_jepa_allpos', '--resume-from', RESUME_CHECKPOINT])
else:
    print('Resume skipped: RESUME_CHECKPOINT is None')


Resume skipped: RESUME_CHECKPOINT is None
